In [1]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile
from tqdm import tqdm
import threading
import time

# ======================
# Device
# ======================
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

# ======================
# Load
# ======================
model_id = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id).to(device)
model.eval()

# ======================
# Inputs
# ======================
text_prompt = ["wolf howl"]
inputs = processor(text=text_prompt, padding=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# ======================
# Duration control
# ======================
target_steps = 200  # this is "new tokens"/audio-code steps for MusicGen

# A "fake" progress bar based on elapsed time (since streamer may not work)
# Calibrate roughly: musicgen-small on MPS often takes tens of seconds.
pbar = tqdm(total=target_steps, desc="Generating (approx)", unit=" step", dynamic_ncols=True)
stop_flag = {"done": False}

def progress_thread():
    # Update slowly while generation runs; doesn't reflect real steps, but doesn't lie about completion either.
    i = 0
    while not stop_flag["done"]:
        time.sleep(0.25)
        if i < target_steps:
            pbar.update(1)
            i += 1

t = threading.Thread(target=progress_thread, daemon=True)
t.start()

with torch.no_grad():
    audio_values = model.generate(
        **inputs,
        max_new_tokens=target_steps,
        do_sample=True,
        guidance_scale=3.5,
        temperature=1.0,
        top_k=250,
    )

stop_flag["done"] = True
time.sleep(0.3)
pbar.close()

# ======================
# Save
# ======================
sampling_rate = model.config.audio_encoder.sampling_rate
audio_data = audio_values[0, 0].detach().cpu().numpy()

scipy.io.wavfile.write("musicgen_out.wav", rate=sampling_rate, data=audio_data)
print("saved: musicgen_out.wav", "samples:", audio_data.shape[0], "seconds:", audio_data.shape[0] / sampling_rate)


device: mps


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Generating (approx):  21%|██        | 42/200 [00:10<00:40,  3.86 step/s]

saved: musicgen_out.wav samples: 126080 seconds: 3.94
